# 5. Photometry

In this tutotial we give a introduction how you can extract and visualise the light curves produced by PlatoSim's build-in photometry module. The algorithm implemented into PlatoSim is a optimal binary mask aperture routine which also will be used for the photometric flux extraction of the so-called P5 sample (i.e. a large subset of the PIC) on-board the spacecraft.

</br>

**Pre-processing steps**

We assume that each pixel value $f_{ij}$ is composed of a signal $s_{ij}$, a background $b_{ij}$, and noise component $\epsilon_{ij}$:  

$$ f_{ij} = s_{ij} + b_{ij} + \varepsilon_{ij}$$

The implemented pre-processing steps of PlatoSim are:
- Bias subtraction (using the bias maps)
- Open shutter smearing correction (using the smearing maps)
- Convert from [ADU] to [electrons] using the (FEE and CCD) gain
- Flatfield correction (using PRNU)
- Sky background subtraction (assumed to be known)

Generally all of these steps requires measurements which introduce noise when applying each operation. The bias and smearing maps are in fact estimated from serial prescan and the parallel overscan regions, respectively. The background $b_{ij}$ is assumed to be know exactly (using the background map), but in reality it will be determined from interpolation of measurements across the full FOV. Furthermore, a few steps may be a part of the future PLATO pipeline which are not yet implemented for the image reduction, being:
- CTI correction
- BFE correction
- Jitter and Drift correction



</br>

**Photometry modules**

The Optimal Aperture Photometry (OAP) method is the currenly planned photometry technique to use for stars processed on-board the PLATO spacecraft (due to limited amount of telemetry capabilities). A full description can by found in [Marchiori et al. (2019)](https://arxiv.org/abs/1906.00892)'s paper and in essence it is optimised to provide the lowest Noise-to-Signal Ratio (NSR) needed for detecting the transit signature of exoplanets. The implementation of the OAP depends on the knowledge of the PSF (position and morphology). By default PlatoSim uses inverted PSFs to determine the optimal aperture mask for each requested mask update.

We note that this method not necessarily provides the optimal description for stellar pulsation and variability (seen in the Fourier domain), but for which ultimately will be a trade-off between including as much of the stellar signal while still trying to avoid stellar contamination.

### Setup notebook

In [ ]:
# Alow changes to the PlatoSim code outside this notebook
%load_ext autoreload
%autoreload 2

# To interact with the plot use
%matplotlib notebook

### Imports

In [ ]:
import os
import random
import numpy as np

# PlatoSim
import platosim.plot            as pt
import platosim.utilities       as ut
import platosim.referenceFrames as rf
from platosim.simulation   import Simulation
from platosim.simfile      import SimFile
from platosim.matplotlibrc import setup_notebook
setup_notebook()

In [ ]:
# Set the output location
outputDir = os.getcwd()

---
## 5.1 - Include photometry in your simulations
---

We start by generating a simulaition object:

In [ ]:
# Set up a Simulation object
outputFileName = "output_example1"
sim = Simulation(outputFileName, outputDir=outputDir)

PlatoSim's photometry module needs to be activated, and a *photometry file* with the index of the star(s) for which photmetry is requested, needs to be saved. The latter can be generated using `sim.createPhotometryTargetFile()` as done here:

In [ ]:
# Automatic photometry file creation
starID = [0, 1, 2, 3, 4]
photometryFile = outputDir + "/photometry_example1.txt"
sim.createPhotometryFile(starID, photometryFile)

Just like creating a star catalogue or a variable source file, the photometry module above automatically activates the photometry module and sets the newly created file to YAML tree needed. Alternatively, you can alter these entries with:

In [ ]:
# Activate photometry
sim["Photometry/IncludePhotometry"] = True
sim["Photometry/TargetFileName"]    = photometryFile

We place a single target in a small subfield:

In [ ]:
# Select subfield size and location
sim["SubField/NumColumns"]      = 10
sim["SubField/NumRows"]         = 10
sim["SubField/ZeroPointRow"]    = 3000
sim["SubField/ZeroPointColumn"] = 3000

# Define catalogue
row = np.array([2.0, 2.25, 5.0, 7.5, 7.25]) + sim["SubField/ZeroPointRow"]
col = np.array([2.5, 7.75, 5.0, 7.5, 2.5]) + sim["SubField/ZeroPointColumn"]
mag = np.array([11.0, 11.0, 11.0, 11.0, 11.0])

# Automatic catalogue file creation
starcatFile = outputDir + "/starcat_example1.txt"
sim.createStarCatalogFileFromPixelCoordinates(row, col, mag, starID, starcatFile)

Since the photometric extraction already uses all the simulated meta- and calibration data under the hood, in this example there is no need to save this to the HDF5 output file. This also significantly reduced the execution time, so always consider which data products you need to save. We here deactivate all output writing and add a few data products. 

In [ ]:
# Turn off saving 
sim.turnOffAllOutput()

# Control HDF5
sim["ControlHDF5Content/GroupByExposure"]    = True
sim["ControlHDF5Content/WritePixelMaps"]     = True
sim["ControlHDF5Content/WriteStarPositions"] = True
sim["ControlHDF5Content/WriteCosmics"]       = True

Let's simulate a time series with a duration of a 1 day at a cadence of 25 seconds:

In [ ]:
sim["ObservingParameters/NumExposures"] = int(86400/25.)

In [ ]:
# Run the simulation
simfile = sim.run(removeOutputFile=True, executionTime=True)

---
## 5.2 - Extract photometry from HDF5
---

Like before to extract information from the HDF5 output file we need to use either the simulation object created with `simulation.run()` or using the class `SimFile`. We suggest to use the latter since the output file this way can alway be fetched fast and you avoid using a simulation object that may have undergone some (unexpected changes). Compared to the other utilities of `SimFile` functions extracting the light curve (time and flux) is returning [Pandas Data Frames](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html) since these are much more convenient to work with for post-processing. First let's fetch the HDF5 output file generated from the previous section:

In [ ]:
f = SimFile(outputFileName + ".hdf5")

### The aperture mask

We start looking at the simulated pixel data and the mask used to extract the photometry:

`getMaskUpdateEvents()`: We can fetch exposure for when the mask has been updated using:

In [ ]:
updates = f.getMaskUpdateEvents()
updates

`getApertureMask()`: This function can be used to fetch the photometric aperture mask used to extract the light curve. We note that this has to be done individually per star. Here we use the mask updates events to get the mask used:

In [ ]:
row, col, update, npix, NSR = f.getApertureMask(starID=starID[0], imageNr=0)
row, col, update, npix, NSR

If the pixel images also have been saved to the HDF5 file (like in this example), we can visualize the used mask for a given exposure with the function `showImage()` and parse the argument `showStarPositions=True` and `showMaskOfStarID=starID` to show the stellar positions together with the aperture mask (here we show for `starID=1`):

In [ ]:
fig, ax = f.showImage(imageNr=False, imgScale="percentile", clip=1,
                      figsize=(6,6), fontSize=15, 
                      showStarPositions=True, showMaskOfStarID=2,
                      colorBar=True, showGrid=True)

### The light curve

`getTime()`: Get the time column simulated:

In [ ]:
f.getTime(df=True).head()

`getFlux()`: Get the flux column for either a specific star or all stars simulated:

In [ ]:
# Fetch flux from first star
f.getFlux(starID[0], df=True).head()

In [ ]:
# Fetch flux from all stars (note the automatic indicing)
f.getFlux(starID, df=True).head()

`getLightCurve()`: Get the light curve of a specific star ID. The photometry module return the parameter `fluxType` which can be either the `estimated` or `input` flux (with the former being the default if no argument is parsed). Let's extract the light curve for the first star:

In [ ]:
df = f.getLightCurve(starID[0], fluxType="estimated", df=True)
df.head()

`plotLightCurve()`: We can also easily visualise the light curve with:

In [ ]:
fig, ax = f.plotLightCurve(starID[0], timeUnit='hour', fluxUnit='e/s');

You can play around with the above and see how the aperture mask for the different stars tries to avoid flux leakage from the neighboring star. Note also that we have placed the three stars (of equal magnitudes) at different intra-pixel positions to illustrate how flux morphology changes for a single realisation of the PSF in the focal plane array.  

### Outliers contaminating the mask

The above time series shows some clear outliers which (most likely) are due to contamination of incident cosmic particles. Except for the core stellar sample of PLATO, the information about cosmic hits will only be indentifiable using the raw pixel data, however, we here show to you can indentify which images that include a cosmic ray hit:

In [ ]:
dex = f.getCosmicsWithinApertureMask(starID=2)
df0 = df.iloc[dex]

In [ ]:
fig, ax = f.plotLightCurve(starID[0], timeUnit='hour', fluxUnit='e/s')
ax.plot(df0.time/3600, df0.flux, 'rx', alpha=0.5);

You can for example use this functionality to compare the efficiency of your own outlier detection algorithm(s). 

---
## 5.3 - Photometry of a variable source
---

We here return to the example of the previous notebook on injecting variable source signals into your simulations. The following code block is simple a copy of the previous notebook:

In [ ]:
# Set up a Simulation object
sim = Simulation("output_example3", outputDir=outputDir)

# Load variable source
starID = [0, 1, 2]
inputDir = os.getenv("PLATO_PROJECT_HOME") + "/inputfiles"
variableSourceFile0 = inputDir + "/varsource_gdor.txt"
variableSourceFile1 = inputDir + "/varsource_algol.txt"
variableSourceFile2 = inputDir + "/varsource_ap.txt"
variableSourceFiles = [variableSourceFile0, variableSourceFile1, variableSourceFile2]

# Create and set variable source file
variableSourceList  = outputDir + "/varlist_example3.txt"
sim.createVariableSourceList(starID, variableSourceFiles, variableSourceList)

# Create and set photometry file (for target star only)
starID = [0]
photometryFile = outputDir + "/photometry_example1.txt"
sim.createPhotometryFile(starID, photometryFile)

# Select subfield size and location
sim["SubField/NumColumns"]      = 8
sim["SubField/NumRows"]         = 8
sim["SubField/ZeroPointRow"]    = 3000
sim["SubField/ZeroPointColumn"] = 3000

# Define catalogue
row = np.array([4.0, 3.1, 5.3]) + sim["SubField/ZeroPointRow"]
col = np.array([4.0, 2.8, 6.2]) + sim["SubField/ZeroPointColumn"]
mag = np.array([10.0, 14.0, 11.0])
starID = [0, 1, 2]

# Create and set stellar catalogue file
starcatFile = outputDir + "/starcat_example3.txt"
sim.createStarCatalogFileFromPixelCoordinates(row, col, mag, starID, starcatFile)

# Save only the eneded output
sim.turnOffAllOutput()
sim["ControlHDF5Content/GroupByExposure"]    = True
sim["ControlHDF5Content/WritePixelMaps"]     = True
sim["ControlHDF5Content/WriteStarPositions"] = True
sim["ControlHDF5Content/WriteCosmics"]       = True

# Save the PSF
sim["ControlHDF5Content/WriteHighResolutionPSF"] = True

# Run simulation for 1 day
nexp = int(86400/25.)
sim["ObservingParameters/NumExposures"] = nexp
simfile = sim.run(removeOutputFile=True, executionTime=True)

Let's fetch the simulation and plot the first image:

In [ ]:
f = SimFile("output_example3.hdf5")

In [ ]:
fig, ax = f.showImage(0, imgScale="percentile", clip=3, 
                      fontSize=20, figsize=(6,6), 
                      showStarPositions='PIC', showMaskOfStarID=0, 
                      colorBar=True, showGrid=True);

Below we show the photometry extracted using the plot function `plotLightCurve()`:

In [ ]:
fig, ax = f.plotLightCurve(starID[0], timeUnit='hour', fluxUnit='ppt')

# Load files
var0 = np.loadtxt(variableSourceFile0)
var1 = np.loadtxt(variableSourceFile1)
var2 = np.loadtxt(variableSourceFile2)

# Show noise-less light curves
for i, var, lab in zip(range(3), [var0, var1, var2], ['gDor', 'EB', 'Ap']):
    time = var[:nexp,0]/3600
    flux = ut.fromMagToRelativeFlux(var[:nexp,1], norm=1e3)
    ax.plot(time, flux, label=lab)
    
# Set limit to zoom in on light curve
tmax = nexp * 25 / 3600
ax.set_ylim(-100, 30)

# Set legend
ax.legend();

It's clear to see that the faint ($V=15$) but close Eclipsing Binary (EB) system (of Algol type) is contaminating the light curve of our gamma Doradus variable star, with a primary eclipse seen at around 5 days. Although the EB is a 14th magnitude star (i.e. around 16 times fainter than our target star), the peak-to-peak variation of 2 mag leaves a clear imprint in the light curve. For computational resources we only simulated the 1 out of 90 days of this variable time series. You can simply increase the number of exposures in the above code snippet to see the full simulated light curve.

Note that if you would like create your own photometry extraction script, we strongly recommend to have a look at the high resolution (diffused) PSF model. The implemented algorithm from [Marchiori et al. (2019)](https://arxiv.org/abs/1906.00892) takes advantage of the knowledge of the PSF which depends highly on the barycentric pixel position of the star and the focal plane location. You can illustrate the exact PSF used in the simulation with:

In [ ]:
fig, ax = f.showPSF("highResPSF", showPixelGrid=True);